In [11]:
%cd /Users/peki/AI/neural-fx/neural-fx
import numpy as np
import audio
import torch
from tqdm import tqdm

DATASET_DIR = '../neural_fx_dataset_float/'

/Users/peki/AI/neural-fx/neural-fx


In [12]:
data = audio.load_dataset(DATASET_DIR, return_tensors=True)

  0%|          | 0/9 [00:00<?, ?it/s]

/Users/peki/AI/neural-fx/neural-fx/audio.py:59: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sample_rate, data = wavfile.read(file_path)
100%|██████████| 9/9 [00:01<00:00,  7.19it/s]


In [136]:
# Model modules

class DilatedConv1d(torch.nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super(DilatedConv1d, self).__init__()
        self.conv = torch.nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, padding=0, dilation=dilation, bias=True)

    def forward(self, x):
        return self.conv(x)

class Conv1d(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super(Conv1d, self).__init__()
        self.conv = torch.nn.Conv1d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

class WaveNetBlock(torch.nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super(WaveNetBlock, self).__init__()
        self.dilated_conv = DilatedConv1d(in_channels, out_channels,kernel_size, dilation)
        self.activation = torch.nn.ReLU()

    def forward(self, x):
        out = self.dilated_conv(x)
        out = self.activation(out)
        return out

class ResidualWaveNetBlock(torch.nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super(ResidualWaveNetBlock, self).__init__()
        self.wavenet_block = WaveNetBlock(in_channels, out_channels, kernel_size, dilation)
        self.conv = Conv1d(out_channels, out_channels)

    def forward(self, x):
        out = self.wavenet_block(x)
        out = self.conv(out)
        return out + x[:, :, -out.size(2):]


class WaveNet(torch.nn.Module):
    def __init__(self, num_layers, num_channels, kernel_size, dilation_array):
        super(WaveNet, self).__init__()
        self.num_layers = num_layers
        self.num_channels = num_channels
        self.kernel_size = kernel_size
        self.dilation_array = dilation_array
        self.receptive_field = self.get_receptive_field()
        self.first_conv = WaveNetBlock(1, self.num_channels, self.kernel_size, self.dilation_array[0])
        self.residual_layers = torch.nn.ModuleList()
        for i in range(1, self.num_layers-1):
            self.residual_layers.append(ResidualWaveNetBlock(self.num_channels, self.num_channels, kernel_size, self.dilation_array[i]))
        self.residual_layers = torch.nn.Sequential(*self.residual_layers)
        self.last_conv = WaveNetBlock(self.num_channels, 1, kernel_size, self.dilation_array[-1])
        self.out = WaveNetBlock(in_channels=1, out_channels=1, kernel_size=1, dilation=1)

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0).unsqueeze(0)
        elif x.dim() == 2:
            x = x.unsqueeze(0)
        if x.shape[-1] < self.receptive_field:
            x = torch.cat([torch.zeros(x.shape[0], 1, self.receptive_field - x.shape[-1], device=x.device), x], dim=-1)
        else:
            x = x[:, :, -self.receptive_field:]
        out = self.first_conv(x)
        out = self.residual_layers(out)
        out = self.last_conv(out)
        out = self.out(out)
        return out

    def get_receptive_field(self):
        return (self.kernel_size - 1) * torch.sum(torch.tensor(self.dilation_array)) + 1
    



In [140]:
# Define WaveNet model
num_layers = 18
num_channels = 3
kernel_size = 3
dilation_array = get_dilation_array(18, 9)

wavenet = WaveNet(num_layers, num_channels, kernel_size, dilation_array)

receptive_field = wavenet.get_receptive_field()
print(receptive_field)
signal = data['DI'].data[:receptive_field+200].unsqueeze(0).unsqueeze(0)
#stack signal to batch
signal = torch.cat([signal, signal, signal], dim=0)
print(signal.shape)
output = wavenet(signal)
print(output.shape)


tensor(2045)
torch.Size([3, 1, 2245])
torch.Size([3, 1, 1])


1023

In [114]:
print(wavenet.get_receptive_field())
print(wavenet.get_receptive_field2())

tensor(1024)
tensor(1033)


In [115]:
wavenet

WaveNet(
  (first_conv): WaveNetBlock(
    (dilated_conv): DilatedConv1d(
      (conv): Conv1d(1, 3, kernel_size=(2,), stride=(1,))
    )
    (activation): ReLU()
  )
  (residual_layers): Sequential(
    (0): ResidualWaveNetBlock(
      (wavenet_block): WaveNetBlock(
        (dilated_conv): DilatedConv1d(
          (conv): Conv1d(3, 3, kernel_size=(2,), stride=(1,), dilation=(2,))
        )
        (activation): ReLU()
      )
      (conv): Conv1d(
        (conv): Conv1d(3, 3, kernel_size=(1,), stride=(1,))
      )
    )
    (1): ResidualWaveNetBlock(
      (wavenet_block): WaveNetBlock(
        (dilated_conv): DilatedConv1d(
          (conv): Conv1d(3, 3, kernel_size=(2,), stride=(1,), dilation=(4,))
        )
        (activation): ReLU()
      )
      (conv): Conv1d(
        (conv): Conv1d(3, 3, kernel_size=(1,), stride=(1,))
      )
    )
    (2): ResidualWaveNetBlock(
      (wavenet_block): WaveNetBlock(
        (dilated_conv): DilatedConv1d(
          (conv): Conv1d(3, 3, kernel_s

In [97]:
MODELS = [
    (10, 10),
    (18, 9),
    (24, 8)
]
def get_dilation_array(num_layers, max_dilation_parameter):
    dilation_array = []
    for i in range(num_layers):
        dilation_array.append(2 ** (i % max_dilation_parameter))
    return dilation_array

print(get_dilation_array(*MODELS[2]))

[1, 2, 4, 8, 16, 32, 64, 128, 1, 2, 4, 8, 16, 32, 64, 128, 1, 2, 4, 8, 16, 32, 64, 128]


<function torch._VariableFunctionsClass.sum>